# Notebook 3 — Subgroup-Aware RoBERTa Baseline

This notebook tests whether annotator subgroup information helps when it is injected explicitly into the model architecture.

Instead of giving the subgroup as text:

```text
[TARGET_SUBGROUP] women
[TWEET] ...
```

this notebook uses:

```text
tweet → RoBERTa CLS embedding
subgroup_id → learned subgroup embedding

[CLS ; subgroup_embedding] → classifier → distribution
```

This checks whether architectural subgroup conditioning is stronger than a plain text-prefix subgroup token.


## 1. Imports and Configuration

In [26]:
import ast
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, confusion_matrix, classification_report
from scipy.spatial.distance import jensenshannon
from scipy.stats import entropy, pearsonr, spearmanr

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

RANDOM_SEED = 42

MODEL_NAME = "roberta-base"
NUM_LABELS = 3
MAX_LENGTH = 192
BATCH_SIZE = 16
EPOCHS = 6
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRAD_CLIP = 1.0
SUBGROUP_DIM = 32
DROPOUT = 0.1

EXPERIMENT = "immigration"  # options: "women", "immigration"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = Path("/home/shayan/Distributional-Hate-Speech-Prediction/data/processed")
OUTPUT_DIR = Path("subgroup_embedding_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Device:", DEVICE)


Device: cuda


In [27]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)


## 2. Load Processed Data

In [28]:
women_df = pd.read_parquet(DATA_DIR / "women_processed.parquet")
immigration_df = pd.read_parquet(DATA_DIR / "immigration_processed.parquet")

print("Women:", women_df.shape)
print("Immigration:", immigration_df.shape)

display(women_df.head(2))


Women: (3869, 12)
Immigration: (3799, 12)


,comment_id,text,split,num_annotations,overall_counts,overall_distribution,entropy,majority_label,expected_label,subgroup_counts,subgroup_label_counts,subgroup_distributions
0,6,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),train,2,"[2.0, 0.0, 0.0]","[1.0, 0.0, 0.0]",0.0,0,0.0,"{'men': 1, 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': 1}","{'men': [1.0, 0.0, 0.0], 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': [1.0, 0.0, 0.0]}","{'men': [1.0, 0.0, 0.0], 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': [1.0, 0.0, 0.0]}"
1,11,"eat my fuck, bitch",validation,2,"[0.0, 1.0, 1.0]","[0.0, 0.5, 0.5]",1.0,1,1.5,"{'men': 1, 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': 1}","{'men': [0.0, 0.0, 1.0], 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': [0.0, 1.0, 0.0]}","{'men': [0.0, 0.0, 1.0], 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': [0.0, 1.0, 0.0]}"


## 3. Expand Comment Rows into Comment–Subgroup Examples

In [29]:
def ensure_dict(value):
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        return ast.literal_eval(value)
    raise TypeError(f"Expected dict or stringified dict, got {type(value)}")


def is_valid_distribution(dist, num_labels: int = NUM_LABELS, tolerance: float = 1e-5) -> bool:
    if dist is None:
        return False
    if len(dist) != num_labels:
        return False
    if any(float(p) < -tolerance for p in dist):
        return False
    return abs(sum(float(p) for p in dist) - 1.0) < tolerance


def expand_subgroup_examples(
    comment_df: pd.DataFrame,
    experiment_name: str,
    allowed_subgroups: list[str] | None = None,
) -> pd.DataFrame:
    rows = []

    for _, row in comment_df.iterrows():
        subgroup_distributions = ensure_dict(row["subgroup_distributions"])
        subgroup_counts = ensure_dict(row["subgroup_counts"])

        for subgroup, target_distribution in subgroup_distributions.items():
            if allowed_subgroups is not None and subgroup not in allowed_subgroups:
                continue

            if not is_valid_distribution(target_distribution):
                continue

            target_distribution = [float(x) for x in target_distribution]

            rows.append({
                "experiment": experiment_name,
                "comment_id": row["comment_id"],
                "split": row["split"],
                "subgroup": subgroup,
                "subgroup_count": int(subgroup_counts.get(subgroup, 0)),
                "text": row["text"],
                "target_distribution": target_distribution,
                "target_majority_label": int(np.argmax(target_distribution)),
                "target_expected_label": float(np.dot(np.arange(NUM_LABELS), target_distribution)),
                "target_entropy": float(entropy(target_distribution, base=2)),
            })

    return pd.DataFrame(rows)


In [30]:
WOMEN_ALLOWED_SUBGROUPS = ["women", "men"]

women_examples = expand_subgroup_examples(
    women_df,
    experiment_name="women",
    allowed_subgroups=WOMEN_ALLOWED_SUBGROUPS,
)

Immigration_ALLOWED_SUBGROUPS = ["neutral","liberal","conservative","slightly_liberal","slightly_conservative","extremely_liberal","extremely_conservative","no_opinion"]
immigration_examples = expand_subgroup_examples(
    immigration_df,
    experiment_name="immigration",
    allowed_subgroups=Immigration_ALLOWED_SUBGROUPS,
)

print("Women examples:", women_examples.shape)
print("Immigration examples:", immigration_examples.shape)

display(women_examples.head())
display(immigration_examples.head())


Women examples: (7738, 10)
Immigration examples: (9090, 10)


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,target_entropy
0,women,6,train,men,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0,0.0
1,women,6,train,women,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0,0.0
2,women,11,validation,men,1,"eat my fuck, bitch","[0.0, 0.0, 1.0]",2,2.0,0.0
3,women,11,validation,women,1,"eat my fuck, bitch","[0.0, 1.0, 0.0]",1,1.0,0.0
4,women,26,train,men,2,I'd love to rip those panties off and shove my fat cock in her ass.,"[0.5, 0.0, 0.5]",0,1.0,1.0


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,target_entropy
0,immigration,7,test,extremely_liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go st...,"[1.0, 0.0, 0.0]",0,0.0,0.0
1,immigration,7,test,liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go st...,"[0.0, 0.0, 1.0]",2,2.0,0.0
2,immigration,10,train,liberal,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're ...","[0.0, 0.0, 1.0]",2,2.0,0.0
3,immigration,10,train,neutral,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're ...","[0.0, 0.0, 1.0]",2,2.0,0.0
4,immigration,10,train,slightly_conservative,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're ...","[1.0, 0.0, 0.0]",0,0.0,0.0


## 4. Select Experiment and Create Subgroup IDs

In [31]:
if EXPERIMENT == "women":
    model_df = women_examples.copy()
elif EXPERIMENT == "immigration":
    model_df = immigration_examples.copy()
else:
    raise ValueError("EXPERIMENT must be 'women' or 'immigration'.")

subgroup_to_id = {
    subgroup: idx
    for idx, subgroup in enumerate(sorted(model_df["subgroup"].unique()))
}

id_to_subgroup = {
    idx: subgroup
    for subgroup, idx in subgroup_to_id.items()
}

model_df["subgroup_id"] = model_df["subgroup"].map(subgroup_to_id)

print("Subgroup mapping:")
print(subgroup_to_id)

display(pd.crosstab(model_df["split"], model_df["subgroup"]))
display(model_df.head())


Subgroup mapping:
{'conservative': 0, 'extremely_conservative': 1, 'extremely_liberal': 2, 'liberal': 3, 'neutral': 4, 'no_opinion': 5, 'slightly_conservative': 6, 'slightly_liberal': 7}


subgroup,conservative,extremely_conservative,extremely_liberal,liberal,neutral,no_opinion,slightly_conservative,slightly_liberal
split,,,,,,,,
test,163,46,197,308,257,46,154,213
train,739,252,907,1407,1027,206,756,1066
validation,157,65,181,311,228,37,169,198


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,target_entropy,subgroup_id
0,immigration,7,test,extremely_liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go st...,"[1.0, 0.0, 0.0]",0,0.0,0.0,2
1,immigration,7,test,liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go st...,"[0.0, 0.0, 1.0]",2,2.0,0.0,3
2,immigration,10,train,liberal,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're ...","[0.0, 0.0, 1.0]",2,2.0,0.0,3
3,immigration,10,train,neutral,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're ...","[0.0, 0.0, 1.0]",2,2.0,0.0,4
4,immigration,10,train,slightly_conservative,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're ...","[1.0, 0.0, 0.0]",0,0.0,0.0,6


In [32]:
train_df = model_df[model_df["split"] == "train"].reset_index(drop=True)
val_df = model_df[model_df["split"].isin(["validation", "val", "dev"])].reset_index(drop=True)
test_df = model_df[model_df["split"] == "test"].reset_index(drop=True)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

assert len(train_df) > 0
assert len(val_df) > 0
assert len(test_df) > 0


Train: (6360, 11)
Val: (1346, 11)
Test: (1384, 11)


In [33]:
print("Training majority-label distribution:")
display(train_df["target_majority_label"].value_counts(normalize=True).sort_index())

print("Test majority-label distribution:")
display(test_df["target_majority_label"].value_counts(normalize=True).sort_index())


Training majority-label distribution:


target_majority_label
0    0.731604
1    0.070912
2    0.197484
Name: proportion, dtype: float64

Test majority-label distribution:


target_majority_label
0    0.743497
1    0.068642
2    0.187861
Name: proportion, dtype: float64

## 5. Dataset and Dataloader

In [34]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


/home/shayan/Distributional-Hate-Speech-Prediction/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [35]:
class SubgroupEmbeddingDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, tokenizer, max_length: int):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx: int):
        row = self.dataframe.iloc[idx]

        encoding = self.tokenizer(
            row["text"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "subgroup_id": torch.tensor(row["subgroup_id"], dtype=torch.long),
            "target_distribution": torch.tensor(row["target_distribution"], dtype=torch.float),
        }


In [36]:
train_dataset = SubgroupEmbeddingDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = SubgroupEmbeddingDataset(val_df, tokenizer, MAX_LENGTH)
test_dataset = SubgroupEmbeddingDataset(test_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_loader))
{k: v.shape for k, v in batch.items()}


{'input_ids': torch.Size([16, 192]),
 'attention_mask': torch.Size([16, 192]),
 'subgroup_id': torch.Size([16]),
 'target_distribution': torch.Size([16, 3])}

## 6. Subgroup-Aware RoBERTa Model

In [37]:
class SubgroupAwareRoBERTa(nn.Module):
    def __init__(
        self,
        model_name: str,
        num_subgroups: int,
        subgroup_dim: int = 32,
        num_labels: int = 3,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.subgroup_embedding = nn.Embedding(
            num_embeddings=num_subgroups,
            embedding_dim=subgroup_dim,
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size + subgroup_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_labels),
        )

    def forward(self, input_ids, attention_mask, subgroup_id):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        subgroup_embedding = self.subgroup_embedding(subgroup_id)

        combined = torch.cat(
            [cls_embedding, subgroup_embedding],
            dim=1,
        )

        logits = self.classifier(combined)

        return {
            "logits": logits,
            "log_probs": torch.log_softmax(logits, dim=-1),
            "probs": torch.softmax(logits, dim=-1),
        }


In [38]:
model = SubgroupAwareRoBERTa(
    model_name=MODEL_NAME,
    num_subgroups=len(subgroup_to_id),
    subgroup_dim=SUBGROUP_DIM,
    num_labels=NUM_LABELS,
    dropout=DROPOUT,
).to(DEVICE)

criterion = nn.KLDivLoss(reduction="batchmean")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

num_training_steps = len(train_loader) * EPOCHS
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

print("Training steps:", num_training_steps)
print("Warmup steps:", num_warmup_steps)


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training steps: 2388
Warmup steps: 238


## 7. Metrics

In [39]:
EPS = 1e-12


def kl_divergence(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    y_true = np.clip(y_true, EPS, 1.0)
    y_pred = np.clip(y_pred, EPS, 1.0)
    return np.sum(y_true * np.log(y_true / y_pred), axis=1)


def js_divergence(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    values = []
    for true_dist, pred_dist in zip(y_true, y_pred):
        values.append(jensenshannon(true_dist, pred_dist, base=2) ** 2)
    return np.array(values)


def cross_entropy(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    y_pred = np.clip(y_pred, EPS, 1.0)
    return -np.sum(y_true * np.log(y_pred), axis=1)


def expected_scores(distributions: np.ndarray) -> np.ndarray:
    labels = np.arange(distributions.shape[1])
    return distributions @ labels


def entropy_values(distributions: np.ndarray) -> np.ndarray:
    return np.array([entropy(dist, base=2) for dist in distributions])


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    true_labels = np.argmax(y_true, axis=1)
    pred_labels = np.argmax(y_pred, axis=1)

    true_expected = expected_scores(y_true)
    pred_expected = expected_scores(y_pred)

    true_entropy = entropy_values(y_true)
    pred_entropy = entropy_values(y_pred)

    metrics = {
        "kl_mean": float(kl_divergence(y_true, y_pred).mean()),
        "js_mean": float(js_divergence(y_true, y_pred).mean()),
        "cross_entropy_mean": float(cross_entropy(y_true, y_pred).mean()),
        "accuracy": float(accuracy_score(true_labels, pred_labels)),
        "macro_f1": float(f1_score(true_labels, pred_labels, average="macro", zero_division=0)),
        "expected_label_mae": float(mean_absolute_error(true_expected, pred_expected)),
    }

    if len(np.unique(true_entropy)) > 1 and len(np.unique(pred_entropy)) > 1:
        metrics["entropy_pearson"] = float(pearsonr(true_entropy, pred_entropy).statistic)
        metrics["entropy_spearman"] = float(spearmanr(true_entropy, pred_entropy).statistic)
    else:
        metrics["entropy_pearson"] = np.nan
        metrics["entropy_spearman"] = np.nan

    return metrics


## 8. Training Helpers

In [40]:
def run_epoch(model, dataloader, optimizer=None, scheduler=None):
    is_training = optimizer is not None

    model.train() if is_training else model.eval()

    total_loss = 0.0
    all_targets = []
    all_predictions = []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        subgroup_id = batch["subgroup_id"].to(DEVICE)
        targets = batch["target_distribution"].to(DEVICE)

        with torch.set_grad_enabled(is_training):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                subgroup_id=subgroup_id,
            )

            loss = criterion(outputs["log_probs"], targets)

            if is_training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()

                if scheduler is not None:
                    scheduler.step()

        total_loss += loss.item() * input_ids.size(0)
        all_targets.append(targets.detach().cpu().numpy())
        all_predictions.append(outputs["probs"].detach().cpu().numpy())

    avg_loss = total_loss / len(dataloader.dataset)

    y_true = np.vstack(all_targets)
    y_pred = np.vstack(all_predictions)

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = float(avg_loss)

    return metrics, y_true, y_pred


## 9. Train Model

In [41]:
best_val_kl = float("inf")
best_model_path = OUTPUT_DIR / f"{EXPERIMENT}_subgroup_embedding_best_model.pt"

history = []

for epoch in range(1, EPOCHS + 1):

    train_metrics, _, _ = run_epoch(
        model,
        train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
    )

    val_metrics, _, _ = run_epoch(
        model,
        val_loader,
        optimizer=None,
        scheduler=None,
    )

    row = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }

    history.append(row)

    print("=" * 80)
    print(f"Epoch {epoch}/{EPOCHS}")
    print(f"Train KL: {train_metrics['kl_mean']:.4f} | Val KL: {val_metrics['kl_mean']:.4f}")
    print(f"Train JS: {train_metrics['js_mean']:.4f} | Val JS: {val_metrics['js_mean']:.4f}")
    print(f"Val Acc: {val_metrics['accuracy']:.4f} | Val Macro F1: {val_metrics['macro_f1']:.4f}")

    if val_metrics["kl_mean"] < best_val_kl:
        best_val_kl = val_metrics["kl_mean"]
        torch.save(model.state_dict(), best_model_path)
        print(f"Saved new best model to {best_model_path}")

history_df = pd.DataFrame(history)
display(history_df)

history_path = OUTPUT_DIR / f"{EXPERIMENT}_subgroup_embedding_training_history.csv"
history_df.to_csv(history_path, index=False)
print("Saved:", history_path)


Epoch 1/6
Train KL: 0.7403 | Val KL: 0.6263
Train JS: 0.2910 | Val JS: 0.2138
Val Acc: 0.7682 | Val Macro F1: 0.4190
Saved new best model to subgroup_embedding_outputs/immigration_subgroup_embedding_best_model.pt
Epoch 2/6
Train KL: 0.5818 | Val KL: 0.5997
Train JS: 0.2184 | Val JS: 0.2202
Val Acc: 0.7645 | Val Macro F1: 0.4496
Saved new best model to subgroup_embedding_outputs/immigration_subgroup_embedding_best_model.pt
Epoch 3/6
Train KL: 0.5313 | Val KL: 0.6491
Train JS: 0.1988 | Val JS: 0.2130
Val Acc: 0.7764 | Val Macro F1: 0.4306
Epoch 4/6
Train KL: 0.4984 | Val KL: 0.6461
Train JS: 0.1851 | Val JS: 0.2276
Val Acc: 0.7422 | Val Macro F1: 0.4572
Epoch 5/6
Train KL: 0.4584 | Val KL: 0.6575
Train JS: 0.1731 | Val JS: 0.2162
Val Acc: 0.7675 | Val Macro F1: 0.4660
Epoch 6/6
Train KL: 0.4292 | Val KL: 0.6933
Train JS: 0.1626 | Val JS: 0.2145
Val Acc: 0.7593 | Val Macro F1: 0.4522


,epoch,train_kl_mean,train_js_mean,train_cross_entropy_mean,train_accuracy,train_macro_f1,train_expected_label_mae,train_entropy_pearson,train_entropy_spearman,train_loss,val_kl_mean,val_js_mean,val_cross_entropy_mean,val_accuracy,val_macro_f1,val_expected_label_mae,val_entropy_pearson,val_entropy_spearman,val_loss
0,1,0.740342,0.290973,0.772793,0.660063,0.389089,0.636832,0.079085,0.079586,0.740342,0.626330,0.213756,0.660508,0.768202,0.418987,0.451315,0.200699,0.198737,0.626330
1,2,0.581803,0.218381,0.614254,0.773742,0.470078,0.474817,0.151009,0.139074,0.581803,0.599695,0.220171,0.633873,0.764487,0.449589,0.473441,0.149191,0.143194,0.599695
2,3,0.531339,0.198802,0.563790,0.796069,0.495265,0.420757,0.145208,0.133344,0.531339,0.649083,0.212984,0.683261,0.776374,0.430565,0.431182,0.193472,0.187821,0.649083
3,4,0.498364,0.185053,0.530815,0.811321,0.513606,0.383928,0.152331,0.141141,0.498364,0.646108,0.227619,0.680286,0.742199,0.457195,0.486908,0.127166,0.119701,0.646108
4,5,0.458419,0.173055,0.490870,0.819811,0.524674,0.353943,0.160296,0.147619,0.458419,0.657505,0.216162,0.691683,0.767459,0.465985,0.443478,0.142174,0.140163,0.657505
5,6,0.429183,0.162614,0.461634,0.824214,0.529203,0.326574,0.159202,0.150616,0.429183,0.693307,0.214508,0.727485,0.759287,0.452154,0.431760,0.146525,0.142242,0.693307


Saved: subgroup_embedding_outputs/immigration_subgroup_embedding_training_history.csv


## 10. Test Evaluation

In [42]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

test_metrics, y_true_test, y_pred_test = run_epoch(
    model,
    test_loader,
    optimizer=None,
    scheduler=None,
)

display(test_metrics)

metrics_path = OUTPUT_DIR / f"{EXPERIMENT}_subgroup_embedding_test_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(test_metrics, f, indent=2)

print("Saved:", metrics_path)


/tmp/ipykernel_2605/331801052.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))


{'kl_mean': 0.595919132232666,
 'js_mean': 0.22377284264177855,
 'cross_entropy_mean': 0.6203439235687256,
 'accuracy': 0.7702312138728323,
 'macro_f1': 0.4482611909840673,
 'expected_label_mae': 0.4890951956440614,
 'entropy_pearson': 0.060467782201834436,
 'entropy_spearman': 0.05939240864234494,
 'loss': 0.5959191489426386}

Saved: subgroup_embedding_outputs/immigration_subgroup_embedding_test_metrics.json


## 11. Save Predictions

In [43]:
predictions_df = test_df.copy()

predictions_df["true_distribution"] = list(y_true_test)
predictions_df["pred_distribution"] = list(y_pred_test)

predictions_df["pred_majority_label"] = np.argmax(y_pred_test, axis=1)
predictions_df["pred_expected_label"] = expected_scores(y_pred_test)
predictions_df["pred_entropy"] = entropy_values(y_pred_test)

predictions_df["kl"] = kl_divergence(y_true_test, y_pred_test)
predictions_df["js"] = js_divergence(y_true_test, y_pred_test)
predictions_df["cross_entropy"] = cross_entropy(y_true_test, y_pred_test)

display(predictions_df.head())

predictions_path = OUTPUT_DIR / f"{EXPERIMENT}_subgroup_embedding_test_predictions.parquet"
predictions_df.to_parquet(predictions_path, index=False)

print("Saved:", predictions_path)


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,target_entropy,subgroup_id,true_distribution,pred_distribution,pred_majority_label,pred_expected_label,pred_entropy,kl,js,cross_entropy
0,immigration,7,test,extremely_liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go st...,"[1.0, 0.0, 0.0]",0,0.0,0.0,2,"[1.0, 0.0, 0.0]","[0.25388247, 0.10426438, 0.64185315]",2,1.387971,1.252781,1.370884,0.544305,1.370884
1,immigration,7,test,liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go st...,"[0.0, 0.0, 1.0]",2,2.0,0.0,3,"[0.0, 0.0, 1.0]","[0.26930162, 0.10648709, 0.6242113]",2,1.354910,1.278192,0.471266,0.219539,0.471266
2,immigration,66,test,extremely_liberal,1,Get ready to be flogged. India is ready to flog all cow dung eating shameless fucktards,"[0.0, 0.0, 1.0]",2,2.0,0.0,2,"[0.0, 0.0, 1.0]","[0.1363965, 0.076253414, 0.7873502]",2,1.650954,0.946728,0.239082,0.115472,0.239082
3,immigration,66,test,slightly_liberal,1,Get ready to be flogged. India is ready to flog all cow dung eating shameless fucktards,"[1.0, 0.0, 0.0]",0,0.0,0.0,7,"[1.0, 0.0, 0.0]","[0.13818295, 0.0774601, 0.78435695]",2,1.646174,0.955272,1.979177,0.696452,1.979177
4,immigration,108,test,conservative,1,"I am secretly hoping he is simply saying that to deter them from coming, he keeps suspending the Haiti deportations but look what happened when he threatened to do it the first time, we got floode...","[0.0, 0.0, 1.0]",2,2.0,0.0,0,"[0.0, 0.0, 1.0]","[0.33786228, 0.09525139, 0.5668863]",2,1.229024,1.316230,0.567596,0.260304,0.567597


Saved: subgroup_embedding_outputs/immigration_subgroup_embedding_test_predictions.parquet


## 12. Diagnostic Evaluation

In [44]:
true_labels = np.argmax(y_true_test, axis=1)
pred_labels = np.argmax(y_pred_test, axis=1)

print("Confusion matrix:")
print(confusion_matrix(true_labels, pred_labels, labels=[0, 1, 2]))

print("\nClassification report:")
print(classification_report(true_labels, pred_labels, labels=[0, 1, 2], zero_division=0))

print("\nPredicted label distribution:")
display(pd.Series(pred_labels).value_counts(normalize=True).sort_index())

print("\nTrue label distribution:")
display(pd.Series(true_labels).value_counts(normalize=True).sort_index())


Confusion matrix:
[[953   0  76]
 [ 72   0  23]
 [147   0 113]]

Classification report:
              precision    recall  f1-score   support

           0       0.81      0.93      0.87      1029
           1       0.00      0.00      0.00        95
           2       0.53      0.43      0.48       260

    accuracy                           0.77      1384
   macro avg       0.45      0.45      0.45      1384
weighted avg       0.70      0.77      0.73      1384


Predicted label distribution:


0    0.846821
2    0.153179
Name: proportion, dtype: float64


True label distribution:


0    0.743497
1    0.068642
2    0.187861
Name: proportion, dtype: float64

In [45]:
print("Mean KL by true majority label:")
display(
    predictions_df
    .groupby("target_majority_label")
    .agg(
        n=("comment_id", "count"),
        mean_kl=("kl", "mean"),
        mean_js=("js", "mean"),
        mean_target_entropy=("target_entropy", "mean"),
        mean_pred_entropy=("pred_entropy", "mean"),
    )
)

print("Average predicted distribution by true majority label:")
for label in [0, 1, 2]:
    subset = predictions_df[predictions_df["target_majority_label"] == label]
    avg_pred = np.vstack(subset["pred_distribution"].to_numpy()).mean(axis=0)
    print(label, avg_pred)


Mean KL by true majority label:


,n,mean_kl,mean_js,mean_target_entropy,mean_pred_entropy
target_majority_label,,,,,
0,1029,0.273940,0.120254,0.035754,0.736307
1,95,2.434744,0.766723,0.052632,0.983239
2,260,1.198336,0.435082,0.026836,1.155091


Average predicted distribution by true majority label:
0 [0.8006931  0.06856313 0.13074408]
1 [0.66367155 0.09044151 0.24588695]
2 [0.4971173  0.10281499 0.40006775]


In [46]:
print("Performance by subgroup:")

subgroup_rows = []

for subgroup, group in predictions_df.groupby("subgroup"):
    y_true = np.vstack(group["true_distribution"].to_numpy())
    y_pred = np.vstack(group["pred_distribution"].to_numpy())

    row = {
        "subgroup": subgroup,
        "n": len(group),
        **compute_metrics(y_true, y_pred),
    }

    subgroup_rows.append(row)

subgroup_metrics_df = pd.DataFrame(subgroup_rows).sort_values("kl_mean")
display(subgroup_metrics_df)

subgroup_metrics_path = OUTPUT_DIR / f"{EXPERIMENT}_subgroup_embedding_subgroup_metrics.csv"
subgroup_metrics_df.to_csv(subgroup_metrics_path, index=False)
print("Saved:", subgroup_metrics_path)


Performance by subgroup:


,subgroup,n,kl_mean,js_mean,cross_entropy_mean,accuracy,macro_f1,expected_label_mae,entropy_pearson,entropy_spearman
1,extremely_conservative,46,0.464958,0.188086,0.475504,0.847826,0.538095,0.446751,0.097175,0.072986
0,conservative,163,0.486455,0.184226,0.498386,0.828221,0.510131,0.403926,-0.002339,0.013943
5,no_opinion,46,0.521677,0.192997,0.540597,0.826087,0.540932,0.413009,-0.132310,-0.093888
3,liberal,308,0.566207,0.211887,0.606469,0.788961,0.463079,0.461034,0.092695,0.095715
6,slightly_conservative,154,0.592981,0.225481,0.604540,0.740260,0.406450,0.505752,-0.069140,-0.067726
2,extremely_liberal,197,0.634325,0.239507,0.655042,0.751269,0.434795,0.520534,0.030132,0.035499
7,slightly_liberal,213,0.656711,0.246917,0.673856,0.727700,0.405888,0.531774,0.089977,0.079596
4,neutral,257,0.659621,0.242729,0.693043,0.754864,0.421328,0.528488,0.109201,0.102617


Saved: subgroup_embedding_outputs/immigration_subgroup_embedding_subgroup_metrics.csv


## 13. Interpretation Guide

Compare this notebook against the old subgroup-token baseline.

Main question:

```text
Does explicit subgroup embedding improve over subgroup-as-text?
```

Look especially at:

- overall KL / JS
- predicted label distribution
- label-1 recall
- label-2 recall
- mean KL by true majority label
- performance by subgroup

If this model improves, then annotator identity is useful but needed stronger architectural conditioning.
If it does not improve, then subgroup identity may not be a strong signal in this dataset.


In [47]:
@torch.no_grad()
def predict_distribution(text, subgroup):

    model.eval()

    encoding = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    subgroup_id = torch.tensor(
        [subgroup_to_id[subgroup]],
        dtype=torch.long
    ).to(DEVICE)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        subgroup_id=subgroup_id
    )

    probs = outputs["probs"].cpu().numpy()[0]

    return probs

In [50]:
from scipy.spatial.distance import jensenshannon
import itertools

@torch.no_grad()
def predict_distribution(text, subgroup):
    model.eval()

    encoding = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    subgroup_id = torch.tensor(
        [subgroup_to_id[subgroup]],
        dtype=torch.long
    ).to(DEVICE)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        subgroup_id=subgroup_id
    )

    return outputs["probs"].cpu().numpy()[0]

In [53]:
ideology_groups = list(subgroup_to_id.keys())
print("Ideology groups:", ideology_groups)
counterfactual_rows = []

for _, row in test_df.drop_duplicates("comment_id").iterrows():

    text = row["text"]

    preds_by_group = {
        group: predict_distribution(text, group)
        for group in ideology_groups
    }

    pairwise_js = []

    for group_a, group_b in itertools.combinations(ideology_groups, 2):

        js = jensenshannon(
            preds_by_group[group_a],
            preds_by_group[group_b],
            base=2
        ) ** 2

        pairwise_js.append(js)

    counterfactual_rows.append({
        "comment_id": row["comment_id"],
        "text": text,
        "mean_pairwise_js": float(np.mean(pairwise_js)),
        "max_pairwise_js": float(np.max(pairwise_js)),
    })

cf_ideology_df = pd.DataFrame(counterfactual_rows)

print("Mean ideology counterfactual JS:", cf_ideology_df["mean_pairwise_js"].mean())
print("Median ideology counterfactual JS:", cf_ideology_df["mean_pairwise_js"].median())
print("Mean max-pairwise JS:", cf_ideology_df["max_pairwise_js"].mean())
print("Median max-pairwise JS:", cf_ideology_df["max_pairwise_js"].median())

display(
    cf_ideology_df
    .sort_values("max_pairwise_js", ascending=False)
    .head(30)
)

Ideology groups: ['conservative', 'extremely_conservative', 'extremely_liberal', 'liberal', 'neutral', 'no_opinion', 'slightly_conservative', 'slightly_liberal']


/home/shayan/Distributional-Hate-Speech-Prediction/.venv/lib/python3.12/site-packages/scipy/spatial/distance.py:1290: RuntimeWarning: invalid value encountered in sqrt
  return np.sqrt(js / 2.0)


Mean ideology counterfactual JS: 0.0001891131495166837
Median ideology counterfactual JS: 0.00018486287966485145
Mean max-pairwise JS: 0.000735540284986286
Median max-pairwise JS: 0.0006863790646537047


,comment_id,text,mean_pairwise_js,max_pairwise_js
206,25323,"All the brits still have Ass hurt from the spanking we gave them a couple hundred years ago. In fact, if their food and weather weren't absolute shit we would have already colonized them. America ...",0.000346,0.001698
107,13279,"They better wake up as soon as possible, start deporting these animals, or hell, if you have to, start getting a bit forceful and kill them. These people wanna wipe out the native Europeans? Well,...",0.000375,0.001690
286,33753,"Like, she's FROM Somalia. The shear arrogance to whitemansplain to her, as if he knows her country of origin – THAT SHE FLED FROM – better than her. Lord, that man's a twat.",0.000347,0.001680
55,7693,"They are fucking scum. Foreign, petit bourgeois, fucking adventurers using saudi and american blood money to fuel their violent, psuedo-sexual fantasies of jihad.",0.000341,0.001679
472,44685,"Rose your not a Mother, your a Mother F -- er . Don't encourage Family members to head to our Borders. Go down to Central America and do something Bitch. America can't take in the whole f -ing wor...",0.000352,0.001618
335,38441,Indian cunts and dickheads orgasming over developing the occupied and annexed Kashmir so they can settle there while the Kashmiri natives remain locked down in their houses with the Indian occupat...,0.000312,0.001611
103,12932,"Need better guns there, USA. How can you even think that you could stop a shooter if in most mass shottings with guns most people only get crippled for life instead of killed? lmao fucking america...",0.000362,0.001593
126,16024,"""Just fucking leave!"" and then people leave the country they are suffering in and move to the US. And then you go ""no not here!"" fucking idiot.",0.000341,0.001570
339,38813,Look at all these fuckers citing Pakistanis / Muslims being terrorist but back their corrupt mofo govt to go and kill innocent people,0.000335,0.001553
403,42177,jeezus those pahang players can't score normal goals,0.000348,0.001550


In [54]:
def pairwise_counterfactual_js(group_a, group_b):
    rows = []

    for _, row in test_df.drop_duplicates("comment_id").iterrows():
        text = row["text"]

        pred_a = predict_distribution(text, group_a)
        pred_b = predict_distribution(text, group_b)

        js = jensenshannon(
            pred_a,
            pred_b,
            base=2
        ) ** 2

        rows.append({
            "comment_id": row["comment_id"],
            "text": text,
            "group_a": group_a,
            "group_b": group_b,
            "js": float(js),
            f"pred_{group_a}": pred_a,
            f"pred_{group_b}": pred_b,
        })

    return pd.DataFrame(rows)


extreme_df = pairwise_counterfactual_js(
    "extremely_liberal",
    "extremely_conservative"
)

print("Extreme liberal vs extreme conservative mean JS:", extreme_df["js"].mean())
print("Extreme liberal vs extreme conservative median JS:", extreme_df["js"].median())

display(
    extreme_df
    .sort_values("js", ascending=False)
    .head(30)
)

Extreme liberal vs extreme conservative mean JS: 0.00046581901191864166
Extreme liberal vs extreme conservative median JS: 0.00044870317697456245


,comment_id,text,group_a,group_b,js,pred_extremely_liberal,pred_extremely_conservative
5,459,"Nuke the border area, Nuke Algeria, Nuke Morocco, Nuke the UN",extremely_liberal,extremely_conservative,0.001079,"[0.75625867, 0.1255433, 0.11819805]","[0.7871606, 0.104356326, 0.1084831]"
90,11006,"It's not the same guy, although the German that Upham let go does end up shooting Captain Miller, who dies minutes later.",extremely_liberal,extremely_conservative,0.001066,"[0.8156969, 0.08235335, 0.10194974]","[0.8435845, 0.06609584, 0.09031972]"
119,14837,Fuck Mongolians.,extremely_liberal,extremely_conservative,0.001033,"[0.71452224, 0.106230274, 0.17924747]","[0.7454318, 0.08774732, 0.16682082]"
329,38064,Keep your people in Mexico and let them get shot there! Ipocrita,extremely_liberal,extremely_conservative,0.001000,"[0.7427523, 0.117522106, 0.13972564]","[0.77290916, 0.098165065, 0.12892579]"
401,42023,All those people who choose to fight agast what must be done to save this country is a ENEMY OF THE STATE OF AMERICA.. BECAUSE THIS IS WAR . LIKE IT OR NOT... ENVASION IS ENVASION...ARE YOU OK WIT...,extremely_liberal,extremely_conservative,0.000984,"[0.75950223, 0.113715515, 0.12678227]","[0.78829277, 0.09415923, 0.117548026]"
114,14514,"Tulsi ""We must stop American imperialism and start American genocide""",extremely_liberal,extremely_conservative,0.000975,"[0.76172346, 0.10806075, 0.13021588]","[0.791534, 0.090863764, 0.11760222]"
314,36717,"""We cannot allow all of these people to invade our Country."" URL",extremely_liberal,extremely_conservative,0.000952,"[0.83752626, 0.086385556, 0.07608818]","[0.8626674, 0.06985588, 0.06747667]"
135,17295,If he wanted to kill illegal immigrants then why drive from Dallas to El Paso to do so? There are tons of them here in Dallas. Many Wal Marts in various parts of Dallas would have also been filled...,extremely_liberal,extremely_conservative,0.000934,"[0.81076175, 0.08524902, 0.103989184]","[0.83739394, 0.070009835, 0.09259616]"
504,46080,Answer is no. America has polluted all nations. Back home in croatia id say they are worse than american girls,extremely_liberal,extremely_conservative,0.000914,"[0.7774285, 0.09826851, 0.12430301]","[0.80372655, 0.08031647, 0.11595708]"
556,48664,"this is not Mexico it's the United States you want Mexico then go there, Wake up America we are under attack from the enemy who have infiltrated our own country, if it's war there after than let's...",extremely_liberal,extremely_conservative,0.000908,"[0.8050251, 0.10046838, 0.0945065]","[0.8311431, 0.082807645, 0.08604929]"


In [57]:
from scipy.spatial.distance import jensenshannon
import numpy as np
import pandas as pd
import ast

def valid_dist(x):
    if x is None:
        return False
    try:
        arr = np.array(x, dtype=float)
    except Exception:
        return False
    if arr.shape[0] != 3:
        return False
    if np.isnan(arr).any():
        return False
    if arr.sum() <= 0:
        return False
    return True


rows = []

for _, row in immigration_df.iterrows():

    subgroup_dists = row["subgroup_distributions"]

    if isinstance(subgroup_dists, str):
        subgroup_dists = ast.literal_eval(subgroup_dists)

    dist_a = subgroup_dists.get("extremely_liberal")
    dist_b = subgroup_dists.get("extremely_conservative")

    if not valid_dist(dist_a) or not valid_dist(dist_b):
        continue

    dist_a = np.array(dist_a, dtype=float)
    dist_b = np.array(dist_b, dtype=float)

    js = jensenshannon(dist_a, dist_b, base=2) ** 2

    rows.append({
        "comment_id": row["comment_id"],
        "text": row["text"],
        "js": float(js),
        "EL_dist": dist_a,
        "EC_dist": dist_b,
    })

true_el_ec_df = pd.DataFrame(rows)

print("N overlapping EL/EC comments:", len(true_el_ec_df))
print("Mean true JS:", true_el_ec_df["js"].mean())
print("Median true JS:", true_el_ec_df["js"].median())

display(
    true_el_ec_df
    .sort_values("js", ascending=False)
    .head(30)
)

N overlapping EL/EC comments: 101
Mean true JS: 0.22725512987854235
Median true JS: 0.0


,comment_id,text,js,EL_dist,EC_dist
100,49332,Send all these asses back !,1.000000,"[1.0, 0.0, 0.0]","[0.0, 1.0, 0.0]"
84,43425,Das what happen when USA is bombing other countries and Killing innosent peoples .. now try to stop this one. Kongo for life and Afrika for life..,1.000000,"[1.0, 0.0, 0.0]","[0.0, 0.0, 1.0]"
59,30714,Your prime minister is a corrupt racist scumbag who hates Arabs and Palestinians though. You should go back to Israel and fix it so that my tax dollars can stop being wasted on bullets used to mur...,1.000000,"[1.0, 0.0, 0.0]","[0.0, 0.0, 1.0]"
64,32928,Thread reeks of jealous pakis and racist brits. Massive DUB URL,1.000000,"[0.0, 1.0, 0.0]","[1.0, 0.0, 0.0]"
69,39456,"Motherfuckers stupid retards Abwehr degenerate German European Israeli ""Jews"" which have nothing to do with the real Hebrews decided to hijack the Western history and CIVILIZATION! - Google Search...",1.000000,"[0.0, 1.0, 0.0]","[0.0, 0.0, 1.0]"
72,40275,I wish the Natives had the same energy and hate these evil folks that's metally broken in Amerikkka have and had when chris brought his evil arse to Amerikkka. We wouldn't be having this conversat...,1.000000,"[0.5, 0.0, 0.5]","[0.0, 1.0, 0.0]"
74,40531,American scum is already attacking Iran.,1.000000,"[1.0, 0.0, 0.0]","[0.0, 0.0, 1.0]"
14,18346,">I am sorry did i hurt your white pride ? In what world does India have more rape per capita than USA ? I guess some people judge humans based on skin color. I'm Pakistani, and I also believe ther...",1.000000,"[1.0, 0.0, 0.0]","[0.0, 1.0, 0.0]"
13,18268,You are really fucking stupid you ignorant low iq american... have you been outside your country? You sound so fucking stupid they should put you down cunt,1.000000,"[1.0, 0.0, 0.0]","[0.0, 0.0, 1.0]"
81,42851,Deport all wops back to the old country.,1.000000,"[0.0, 0.0, 1.0]","[1.0, 0.0, 0.0]"


In [59]:
def majority_label(dist):
    return int(np.argmax(np.array(dist, dtype=float)))


true_el_ec_df["EL_label"] = true_el_ec_df["EL_dist"].apply(majority_label)
true_el_ec_df["EC_label"] = true_el_ec_df["EC_dist"].apply(majority_label)

true_el_ec_df["label_pair"] = (
    true_el_ec_df["EL_label"].astype(str)
    + "-"
    + true_el_ec_df["EC_label"].astype(str)
)

display(
    true_el_ec_df
    .groupby("label_pair")
    .agg(
        n=("comment_id", "count"),
        mean_js=("js", "mean"),
        median_js=("js", "median"),
    )
    .sort_values("n", ascending=False)
)

,n,mean_js,median_js
label_pair,,,
0-0,51,0.023588,0.0
2-2,28,0.026777,0.0
0-2,7,1.000000,1.0
0-1,5,1.000000,1.0
2-0,4,1.000000,1.0
1-2,2,1.000000,1.0
2-1,2,1.000000,1.0
1-0,1,1.000000,1.0
1-1,1,0.000000,0.0


In [60]:
display(
    true_el_ec_df
    .query("js > 0")
    .groupby("label_pair")
    .agg(
        n=("comment_id", "count"),
        mean_js=("js", "mean"),
        median_js=("js", "median"),
    )
    .sort_values("n", ascending=False)
)

,n,mean_js,median_js
label_pair,,,
2-2,12,0.062480,0.025329
0-0,7,0.171858,0.155639
0-2,7,1.000000,1.000000
0-1,5,1.000000,1.000000
2-0,4,1.000000,1.000000
1-2,2,1.000000,1.000000
2-1,2,1.000000,1.000000
1-0,1,1.000000,1.000000


In [61]:
true_el_ec_df["involves_label_1"] = (
    (true_el_ec_df["EL_label"] == 1)
    | (true_el_ec_df["EC_label"] == 1)
)

display(
    true_el_ec_df
    .groupby("involves_label_1")
    .agg(
        n=("comment_id", "count"),
        mean_js=("js", "mean"),
        median_js=("js", "median"),
    )
)

,n,mean_js,median_js
involves_label_1,,,
False,90,0.143920,0.0
True,11,0.909091,1.0


In [62]:
display(
    true_el_ec_df
    .query("js > 0")
    .groupby("involves_label_1")
    .agg(
        n=("comment_id", "count"),
        mean_js=("js", "mean"),
        median_js=("js", "median"),
    )
)

,n,mean_js,median_js
involves_label_1,,,
False,30,0.431759,0.179214
True,10,1.000000,1.000000


In [63]:
comparison_df = true_el_ec_df.merge(
    extreme_df[[
        "comment_id",
        "js",
        "pred_extremely_liberal",
        "pred_extremely_conservative"
    ]],
    on="comment_id",
    suffixes=("_true", "_model")
)

comparison_df = comparison_df.rename(columns={
    "js_true": "true_js",
    "js_model": "model_js"
})

comparison_df["capture_ratio"] = (
    comparison_df["model_js"] /
    (comparison_df["true_js"] + 1e-12)
)

display(comparison_df.head())

,comment_id,text,true_js,EL_dist,EC_dist,EL_label,EC_label,label_pair,involves_label_1,model_js,pred_extremely_liberal,pred_extremely_conservative,capture_ratio
0,13695,Yes so every lazy poverty stricken uneducated third world citizen in the entire world from Africa to South America can come here and get welfare and turn America into just as bad of a shithole as ...,0.311278,"[0.5, 0.0, 0.5]","[1.0, 0.0, 0.0]",0,0,0-0,False,0.000342,"[0.6819129, 0.11475086, 0.20333622]","[0.69854206, 0.102396846, 0.19906108]",1.099420e-03
1,20016,Keep the fucking Mexican scum rapists and murderers out of the country and the fucking Arab Islamic terrorists out of the U.S. too!! No Syrian refugees either!! Build a wall and keep all of these ...,0.004567,"[0.08421052992343903, 0.10526315867900848, 0.8105263113975525]","[0.06666667014360428, 0.06666667014360428, 0.8666666746139526]",2,2,2-2,False,0.000227,"[0.20564345, 0.096090235, 0.69826627]","[0.2027107, 0.08656365, 0.71072567]",4.961086e-02
2,20033,"STOP SUCKING THAT FATASS' DICK, YOU NIGGERLOVING KIKE",0.000000,"[0.0, 0.0, 1.0]","[0.0, 0.0, 1.0]",2,2,2-2,False,0.000279,"[0.14841586, 0.080010474, 0.7715737]","[0.14178166, 0.07151637, 0.7867019]",2.794933e+08
3,20062,I've always been proud of the Igbo tribe and our culture. Yesterday made me even more proud. You people are too fine abeg. OgbakoUmuIgboDay,0.000000,"[1.0, 0.0, 0.0]","[1.0, 0.0, 0.0]",0,0,0-0,False,0.000351,"[0.9419692, 0.034675945, 0.023354875]","[0.95169693, 0.028031966, 0.020271135]",3.510777e+08
4,37246,racists b like go back to ur country like ok stop bombing it n maybe i will,0.000000,"[1.0, 0.0, 0.0]","[1.0, 0.0, 0.0]",0,0,0-0,False,0.000381,"[0.58269054, 0.13809553, 0.27921396]","[0.6010187, 0.1241873, 0.27479404]",3.812121e+08


In [64]:
print("Mean true JS:", comparison_df["true_js"].mean())
print("Mean model JS:", comparison_df["model_js"].mean())
print("Mean capture ratio:", comparison_df["capture_ratio"].mean())
print("Median capture ratio:", comparison_df["capture_ratio"].median())

Mean true JS: 0.16448066401642414
Mean model JS: 0.0003310633278506607
Mean capture ratio: 224576822.36447835
Median capture ratio: 211134988.39196873


In [65]:
display(
    comparison_df
    .groupby("label_pair")
    .agg(
        n=("comment_id", "count"),
        mean_true_js=("true_js", "mean"),
        mean_model_js=("model_js", "mean"),
        mean_capture_ratio=("capture_ratio", "mean"),
        median_capture_ratio=("capture_ratio", "median"),
    )
    .sort_values("mean_true_js", ascending=False)
)

,n,mean_true_js,mean_model_js,mean_capture_ratio,median_capture_ratio
label_pair,,,,,
0-1,1,1.000000,0.000283,2.830846e-04,2.830846e-04
0-0,4,0.077820,0.000429,3.435862e+08,3.661449e+08
2-2,3,0.001522,0.000216,1.407567e+08,1.427767e+08


In [66]:
comparison_nonzero = comparison_df[
    comparison_df["true_js"] > 0
]

comparison_nonzero["capture_ratio"] = (
    comparison_nonzero["model_js"]
    /
    comparison_nonzero["true_js"]
)

print(
    "Mean capture ratio:",
    comparison_nonzero["capture_ratio"].mean()
)

print(
    "Median capture ratio:",
    comparison_nonzero["capture_ratio"].median()
)

Mean capture ratio: 0.01699778781665204
Median capture ratio: 0.001099419760366172


/tmp/ipykernel_2605/3760677297.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  comparison_nonzero["capture_ratio"] = (
